# 实践实验：决策树

在本练习中，你将从头实现一个决策树，并把它用于判断蘑菇是否有毒可食的分类任务。

# 目录
- [ 1 - 程序包 ](#1)
- [ 2 -  问题陈述](#2)
- [ 3 - 数据集](#3)
  - [ 3.1 独热编码数据集](#3.1)
- [ 4 - 决策树回顾](#4)
  - [ 4.1  计算熵](#4.1)
    - [ 练习 1](#ex01)
  - [ 4.2  划分数据集](#4.2)
    - [ 练习 2](#ex02)
  - [ 4.3  计算信息增益](#4.3)
    - [ 练习 3](#ex03)
  - [ 4.4  寻找最佳划分](#4.4)
    - [ 练习 4](#ex04)
- [ 5 - 构建树](#5)


<a name="1"></a>
## 1 - 程序包

首先，运行下面的单元格，导入本次作业中会用到的所有程序包。
- [numpy](www.numpy.org) 是 Python 中处理矩阵的基础程序包。
- [matplotlib](http://matplotlib.org) 是一个著名的 Python 绘图库。
- ``utils.py`` 包含本次作业的辅助函数。你不需要修改该文件中的代码。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from public_tests import *

%matplotlib inline

<a name="2"></a>
## 2 -  问题陈述

假设你正在创办一家种植和销售野生蘑菇的公司。
- 由于并非所有蘑菇都能食用，你希望根据蘑菇的物理特征判断它是可食用的还是有毒的
- 你手头有一些可用于这项任务的数据。

你能用这些数据来帮助判断哪些蘑菇可以安全出售吗？

注意：这里使用的数据集仅用于示例说明，不能作为识别可食用蘑菇的指南。



<a name="3"></a>
## 3 - 数据集

你将先加载本任务的数据集。你收集到的数据集如下：

| 菌盖颜色 | 菌柄形状 | 独生 | 可食用 |
|:---------:|:-----------:|:--------:|:------:|
|   褐色   |   渐细  |    是   |    1   |
|   褐色   |  渐粗  |    是   |    1   |
|   褐色   |  渐粗  |    否    |    0   |
|   褐色   |  渐粗  |    否    |    0   |
|   褐色   |   渐细  |    是   |    1   |
|    红色    |   渐细  |    是   |    0   |
|    红色    |  渐粗  |    否    |    0   |
|   褐色   |  渐粗  |    是   |    1   |
|    红色    |   渐细  |    否    |    1   |
|   褐色   |  渐粗  |    否    |    0   |


-  你有 10 个蘑菇样本。对每个样本，你有
    - 三个特征
        - 菌盖颜色（`褐色` 或 `红色`），
        - 菌柄形状（`渐细` 或 `渐粗`），以及
        - 是否独生（`是` 或 `否`）
    - 标签
        - 可食用（`1` 表示是，`0` 表示有毒）

<a name="3.1"></a>
### 3.1 独热编码数据集
为便于实现，我们对特征做了独热编码（把它们转成取值为 0 或 1 的特征）

| 褐色菌盖 | 渐细菌柄形状 | 独生 | 可食用 |
|:---------:|:--------------------:|:--------:|:------:|
|     1     |           1          |     1    |    1   |
|     1     |           0          |     1    |    1   |
|     1     |           0          |     0    |    0   |
|     1     |           0          |     0    |    0   |
|     1     |           1          |     1    |    1   |
|     0     |           1          |     1    |    0   |
|     0     |           0          |     0    |    0   |
|     1     |           0          |     1    |    1   |
|     0     |           1          |     0    |    1   |
|     1     |           0          |     0    |    0   |

因此，
- `X_train` 为每个样本包含三个特征
    - 是否为褐色（取值 `1` 表示菌盖是“褐色”，`0` 表示“红色”）
    - 是否为渐细形状（取值 `1` 表示“渐细菌柄形状”，`0` 表示“渐粗”菌柄形状）
    - 是否独生（取值 `1` 表示“是”，`0` 表示“否”）

- `y_train` 表示蘑菇是否可食用
    - `y = 1` 表示可食用
    - `y = 0` 表示有毒

In [ ]:
X_train = np.array([[1,1,1],[1,0,1],[1,0,0],[1,0,0],[1,1,1],[0,1,1],[0,0,0],[1,0,1],[0,1,0],[1,0,0]])
y_train = np.array([1,1,0,0,1,0,0,1,1,0])

#### 查看变量
让我们更熟悉一下你的数据集。  
- 一个好的起点就是把每个变量打印出来，看看它包含什么。

下面的代码打印 `X_train` 的前几个元素以及该变量的类型。

In [ ]:
print("First few elements of X_train:\n", X_train[:5])
print("Type of X_train:",type(X_train))

现在，我们对 `y_train` 做同样的操作

In [ ]:
print("First few elements of y_train:", y_train[:5])
print("Type of y_train:",type(y_train))

#### 检查变量的维度

熟悉数据的另一种有用方式是查看它的维度。

请打印 `X_train` 和 `y_train` 的形状，看看数据集中有多少训练样本。

In [ ]:
print ('The shape of X_train is:', X_train.shape)
print ('The shape of y_train is: ', y_train.shape)
print ('Number of training examples (m):', len(X_train))

<a name="4"></a>
## 4 - 决策树回顾

在本实践实验中，你将基于所提供的数据集构建一棵决策树。

- 回忆一下，构建决策树的步骤如下：
    - 从根节点包含所有样本开始
    - 计算按所有可能特征划分的信息增益，选出信息增益最高的那个
    - 根据选中的特征划分数据集，并创建树的左、右分支
    - 反复进行划分过程，直到满足停止条件
  
  
- 在本实验中，你将实现以下函数，它们让你能用信息增益最高的特征把一个节点划分为左、右分支
    - 计算某个节点的熵
    - 根据给定特征把某个节点的数据集划分为左、右分支
    - 计算按给定特征划分的信息增益
    - 选择使信息增益最大的特征
    
- 然后我们将用你实现的辅助函数构建一棵决策树，反复进行划分过程直到满足停止条件
    - 在本实验中，我们选择的停止条件是最大深度设为 2

<a name="4.1"></a>
### 4.1  计算熵

首先，你将编写一个名为 `compute_entropy` 的辅助函数，用于计算某个节点的熵（不纯度的度量）。
- 该函数接收一个 numpy 数组（`y`），表示该节点中的样本是可食用（`1`）还是有毒（`0`）

请完成下面的 `compute_entropy()` 函数，使其能够：
* 计算 $p_1$，即可食用样本的比例（即在 `y` 中取值 = `1` 的样本比例）
* 然后熵按下式计算

$$H(p_1) = -p_1 \text{log}_2(p_1) - (1- p_1) \text{log}_2(1- p_1)$$
* 注意
    * 对数以 $2$ 为底
    * 就实现而言，$0\text{log}_2(0) = 0$。也就是说，如果 `p_1 = 0` 或 `p_1 = 1`，把熵设为 `0`
    * 务必检查某个节点的数据不为空（即 `len(y) != 0`）。如果为空，返回 `0`
    
<a name="ex01"></a>
### 练习 1

请按照前面的说明完成 `compute_entropy()` 函数。
    
如果遇到困难，可以查看下面单元格之后给出的提示来帮助你实现。

In [ ]:
# UNQ_C1
# 计分函数：compute_entropy

def compute_entropy(y):
    """计算熵
   
    参数：
       y (ndarray): Numpy 数组，表示某个节点处每个样本是
           可食用（`1`）还是有毒（`0`）
       
    返回：
        entropy (float): 该节点处的熵"""
    # 你需要正确返回以下变量
    entropy = 0.
    
    ### 在此处编写代码 ###
           
    ### 代码编写结束 ###
    
    return entropy

<details>
<summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>


* 计算 `p1`
   * 你可以用 `y[y == 1]` 取出 `y` 中取值为 `1` 的样本子集
   * 你可以用 `len(y)` 得到 `y` 中的样本数
* 计算 `entropy`
   * <a href="https://numpy.org/doc/stable/reference/generated/numpy.log2.html">np.log2</a> 可以让你对 numpy 数组计算以 2 为底的对数
   * 如果 `p1` 的值为 0 或 1，务必把熵设为 `0` 

<details>
      <summary><font size="2" color="darkblue"><b> 点击查看更多提示</b></font></summary>

* 下面是该函数整体实现的结构

```python 
def compute_entropy(y):

    # You need to return the following variables correctly
    entropy = 0.

    ### START CODE HERE ###
    if len(y) != 0:
        # Your code here to calculate the fraction of edible examples (i.e with value = 1 in y)
        p1 =

        # For p1 = 0 and 1, set the entropy to 0 (to handle 0log0)
        if p1 != 0 and p1 != 1:
            # Your code here to calculate the entropy using the formula provided above
            entropy = 
        else:
            entropy = 0. 
    ### END CODE HERE ###        

    return entropy
```


如果仍然卡住，可以查看下面给出的提示，弄清如何计算 `p1` 和 `entropy`。

<details>
      <summary><font size="2" color="darkblue"><b>计算 p1 的提示</b></font></summary>
       &emsp; &emsp; 你可以这样计算 p1：<code>p1 = len(y[y == 1]) / len(y) </code>
</details>

 <details>
      <summary><font size="2" color="darkblue"><b>计算 entropy 的提示</b></font></summary>
      &emsp; &emsp; 你可以这样计算 entropy：<code>entropy = -p1 * np.log2(p1) - (1 - p1) * np.log2(1 - p1)</code>
</details>

</details>

</details>




你可以运行下面的测试代码，检查你的实现是否正确：

In [ ]:
# 计算根节点（即包含所有样本）处的熵
# 由于我们有 5 个可食用和 5 个不可食用的蘑菇，熵应当为 1"

print("Entropy at root node: ", compute_entropy(y_train)) 

# 单元测试
compute_entropy_test(compute_entropy)

**预期输出**：
<table>
  <tr>
    <td> <b>根节点处的熵：<b> 1.0 </td> 
  </tr>
</table>

<a name="4.2"></a>
### 4.2  划分数据集

接下来，你将编写一个名为 `split_dataset` 的辅助函数，它接收某个节点的数据和用于划分的特征，把数据划分到左、右分支。本实验稍后，你将实现代码来计算这次划分有多好。

- 该函数接收训练数据、该节点数据点的索引列表，以及用于划分的特征。
- 它划分数据，并返回左分支和右分支的索引子集。
- 例如，假设我们从根节点开始（因此 `node_indices = [0,1,2,3,4,5,6,7,8,9]`），并选择按特征 `0` 划分，即样本是否为褐色菌盖。
    - 函数的输出就是 `left_indices = [0,1,2,3,4,7,9]` 和 `right_indices = [5,6,8]`
    
| 索引 | 褐色菌盖 | 渐细菌柄形状 | 独生 | 可食用 |
|:-----:|:---------:|:--------------------:|:--------:|:------:|
|   0   |     1     |           1          |     1    |    1   |
|   1   |     1     |           0          |     1    |    1   |
|   2   |     1     |           0          |     0    |    0   |
|   3   |     1     |           0          |     0    |    0   |
|   4   |     1     |           1          |     1    |    1   |
|   5   |     0     |           1          |     1    |    0   |
|   6   |     0     |           0          |     0    |    0   |
|   7   |     1     |           0          |     1    |    1   |
|   8   |     0     |           1          |     0    |    1   |
|   9   |     1     |           0          |     0    |    0   |

<a name="ex02"></a>
### 练习 2

请完成下面所示的 `split_dataset()` 函数

- 对 `node_indices` 中的每个索引
    - 如果 `X` 在该索引处、该特征上的取值为 `1`，把该索引加入 `left_indices`
    - 如果 `X` 在该索引处、该特征上的取值为 `0`，把该索引加入 `right_indices`

如果遇到困难，可以查看下面单元格之后给出的提示来帮助你实现。

In [ ]:
# UNQ_C2
# 计分函数：split_dataset

def split_dataset(X, node_indices, feature):
    """把给定节点处的数据划分到
    左分支和右分支
   
    参数：
        X (ndarray):             形状为 (n_samples, n_features) 的数据矩阵
        node_indices (ndarray):  包含有效索引的列表。即本步骤中要考虑的样本。
        feature (int):           用于划分的特征索引
   
    返回：
        left_indices (ndarray): 特征取值 == 1 的索引
        right_indices (ndarray): 特征取值 == 0 的索引"""
    
    # 你需要正确返回以下变量
    left_indices = []
    right_indices = []
    
    ### 在此处编写代码 ###
           
    ### 代码编写结束 ###
        
    return left_indices, right_indices

<details>
<summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>


* 下面是该函数整体实现的结构

```python 
def split_dataset(X, node_indices, feature):

    # You need to return the following variables correctly
    left_indices = []
    right_indices = []

    ### START CODE HERE ###
    # Go through the indices of examples at that node
    for i in node_indices:   
        if # Your code here to check if the value of X at that index for the feature is 1
            left_indices.append(i)
        else:
            right_indices.append(i)
    ### END CODE HERE ###

return left_indices, right_indices
```

<details>
      <summary><font size="2" color="darkblue"><b> 点击查看更多提示</b></font></summary>

条件为 <code> if X[i][feature] == 1:</code>。

</details>

</details>




现在，让我们用下面的代码块检查你的实现。让我们试着在根节点处划分数据集，正如上面所讨论的，根节点在特征 0（褐色菌盖）上包含所有样本

In [ ]:
root_indices = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

# 可以随意调整这些变量
# 数据集只有三个特征，所以该取值可以是 0（褐色菌盖）、1（渐细菌柄形状）或 2（独生）
feature = 0

left_indices, right_indices = split_dataset(X_train, root_indices, feature)

print("Left indices: ", left_indices)
print("Right indices: ", right_indices)

# 单元测试
split_dataset_test(split_dataset)

**预期输出**：

```
Left indices:  [0, 1, 2, 3, 4, 7, 9]
Right indices:  [5, 6, 8]
```


<a name="4.3"></a>
### 4.3  计算信息增益

接下来，你将编写一个名为 `information_gain` 的函数，它接收训练数据、某个节点的索引，以及用于划分的特征，返回这次划分的信息增益。

<a name="ex03"></a>
### 练习 3

请完成下面所示的 `compute_information_gain()` 函数，计算

$$\text{Information Gain} = H(p_1^\text{node})- (w^{\text{left}}H(p_1^\text{left}) + w^{\text{right}}H(p_1^\text{right}))$$

其中
- $H(p_1^\text{node})$ 是该节点的熵
- $H(p_1^\text{left})$ 和 $H(p_1^\text{right})$ 是划分得到的左、右分支的熵
- $w^{\text{left}}$ 和 $w^{\text{right}}$ 分别是左、右分支中样本所占的比例

注意：
- 你可以用上面实现的 `compute_entropy()` 函数来计算熵
- 我们提供了一些起始代码，使用你上面实现的 `split_dataset()` 函数来划分数据集

如果遇到困难，可以查看下面单元格之后给出的提示来帮助你实现。

In [ ]:
# UNQ_C3
# 计分函数：compute_information_gain

def compute_information_gain(X, y, node_indices, feature):
    
    """计算按给定特征划分该节点所获得的信息增益
   
    参数：
        X (ndarray):            形状为 (n_samples, n_features) 的数据矩阵
        y (array like):         含 n_samples 个目标变量的列表或 ndarray
        node_indices (ndarray): 包含有效索引的列表。即本步骤中要考虑的样本。
  
    返回：
        cost (float):        计算得到的代价"""    
    # 划分数据集
    left_indices, right_indices = split_dataset(X, node_indices, feature)
    
    # 一些有用的变量
    X_node, y_node = X[node_indices], y[node_indices]
    X_left, y_left = X[left_indices], y[left_indices]
    X_right, y_right = X[right_indices], y[right_indices]
    
    # 你需要正确返回以下变量
    information_gain = 0
    
    ### 在此处编写代码 ###
    
    # 权重
    
    #加权熵
     
    #信息增益
    
    ### 代码编写结束 ###
    
    return information_gain

<details>
<summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>


* 下面是该函数整体实现的结构

```python 
def compute_information_gain(X, y, node_indices, feature):
    # Split dataset
    left_indices, right_indices = split_dataset(X, node_indices, feature)

    # Some useful variables
    X_node, y_node = X[node_indices], y[node_indices]
    X_left, y_left = X[left_indices], y[left_indices]
    X_right, y_right = X[right_indices], y[right_indices]

    # You need to return the following variables correctly
    information_gain = 0

    ### START CODE HERE ###
    # Your code here to compute the entropy at the node using compute_entropy()
    node_entropy = 
    # Your code here to compute the entropy at the left branch
    left_entropy = 
    # Your code here to compute the entropy at the right branch
    right_entropy = 

    # Your code here to compute the proportion of examples at the left branch
    w_left = 

    # Your code here to compute the proportion of examples at the right branch
    w_right = 

    # Your code here to compute weighted entropy from the split using 
    # w_left, w_right, left_entropy and right_entropy
    weighted_entropy = 

    # Your code here to compute the information gain as the entropy at the node
    # minus the weighted entropy
    information_gain = 
    ### END CODE HERE ###  

    return information_gain
```

如果仍然卡住，查看下面的提示。

<details>
      <summary><font size="2" color="darkblue"><b> 计算各熵值的提示</b></font></summary>

<code>node_entropy = compute_entropy(y_node)</code><br>
<code>left_entropy = compute_entropy(y_left)</code><br>
<code>right_entropy = compute_entropy(y_right)</code>

</details>

<details>
      <summary><font size="2" color="darkblue"><b>计算 w_left 和 w_right 的提示</b></font></summary>
       <code>w_left = len(X_left) / len(X_node)</code><br>
       <code>w_right = len(X_right) / len(X_node)</code>
</details>

<details>
      <summary><font size="2" color="darkblue"><b>计算 weighted_entropy 的提示</b></font></summary>
       <code>weighted_entropy = w_left * left_entropy + w_right * right_entropy</code>
</details>

<details>
      <summary><font size="2" color="darkblue"><b>计算 information_gain 的提示</b></font></summary>
       <code> information_gain = node_entropy - weighted_entropy</code>
</details>


</details>


现在你可以用下面的单元格检查你的实现，并计算按各个特征划分时的信息增益

In [ ]:
info_gain0 = compute_information_gain(X_train, y_train, root_indices, feature=0)
print("Information Gain from splitting the root on brown cap: ", info_gain0)
    
info_gain1 = compute_information_gain(X_train, y_train, root_indices, feature=1)
print("Information Gain from splitting the root on tapering stalk shape: ", info_gain1)

info_gain2 = compute_information_gain(X_train, y_train, root_indices, feature=2)
print("Information Gain from splitting the root on solitary: ", info_gain2)

# 单元测试
compute_information_gain_test(compute_information_gain)

**预期输出**：

```
Information Gain from splitting the root on brown cap:  0.034851554559677034
Information Gain from splitting the root on tapering stalk shape:  0.12451124978365313
Information Gain from splitting the root on solitary:  0.2780719051126377
```


在根节点按“独生”（特征 = 2）划分得到最大的信息增益。因此，它是在根节点划分的最佳特征。

<a name="4.4"></a>
### 4.4  寻找最佳划分
现在让我们编写一个函数，像上面那样计算每个特征的信息增益，从而找出最佳划分特征，并返回带来最大信息增益的特征

<a name="ex04"></a>
### 练习 4
请完成下面所示的 `get_best_split()` 函数。
- 该函数接收训练数据以及该节点数据点的索引
- 函数的输出是带来最大信息增益的特征
    - 你可以用 `compute_information_gain()` 函数遍历各特征，计算每个特征的信息增益
如果遇到困难，可以查看下面单元格之后给出的提示来帮助你实现。

In [ ]:
# UNQ_C4
# 计分函数：get_best_split

def get_best_split(X, y, node_indices):   
    """返回用于划分节点数据的最优特征和阈值
   
    参数：
        X (ndarray):            形状为 (n_samples, n_features) 的数据矩阵
        y (array like):         含 n_samples 个目标变量的列表或 ndarray
        node_indices (ndarray): 包含有效索引的列表。即本步骤中要考虑的样本。

    返回：
        best_feature (int):     最优划分特征的索引"""    
    
    # 一些有用的变量
    num_features = X.shape[1]
    
    # 你需要正确返回以下变量
    best_feature = -1
    
    ### 在此处编写代码 ###
       
    ### 代码编写结束 ##
   
    return best_feature

<details>
<summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>


* 下面是该函数整体实现的结构


```python 
def get_best_split(X, y, node_indices):   

    # Some useful variables
    num_features = X.shape[1]

    # You need to return the following variables correctly
    best_feature = -1

    ### START CODE HERE ###
    max_info_gain = 0

    # Iterate through all features
    for feature in range(num_features): 

        # Your code here to compute the information gain from splitting on this feature
        info_gain = 

        # If the information gain is larger than the max seen so far
        if info_gain > max_info_gain:  
            # Your code here to set the max_info_gain and best_feature
            max_info_gain = 
            best_feature = 
    ### END CODE HERE ##    

return best_feature
```

如果仍然卡住，查看下面的提示。

<details>
      <summary><font size="2" color="darkblue"><b> 计算 info_gain 的提示</b></font></summary>

<code>info_gain = compute_information_gain(X, y, node_indices, feature)</code>
</details>

<details>
      <summary><font size="2" color="darkblue"><b>更新 max_info_gain 和 best_feature 的提示</b></font></summary>
       <code>max_info_gain = info_gain</code><br>
       <code>best_feature = feature</code>
</details>
</details>


现在，让我们用下面的单元格检查你的函数实现。

In [ ]:
best_feature = get_best_split(X_train, y_train, root_indices)
print("Best feature to split on: %d" % best_feature)

# 单元测试
get_best_split_test(get_best_split)

正如我们上面看到的，该函数返回：在根节点划分的最佳特征是特征 2（“独生”）

<a name="5"></a>
## 5 - 构建树

在本节中，我们用你上面实现的函数，通过不断选择最佳划分特征来生成一棵决策树，直到达到停止条件（最大深度为 2）。

这一部分你不需要实现任何内容。

In [ ]:
# 不计分
tree = []

def build_tree_recursive(X, y, node_indices, branch_name, max_depth, current_depth):
    """使用递归算法构建一棵树，在每个节点把数据集划分为 2 个子组。
    本函数只是打印这棵树。
   
    参数：
        X (ndarray):            形状为 (n_samples, n_features) 的数据矩阵
        y (array like):         含 n_samples 个目标变量的列表或 ndarray
        node_indices (ndarray): 包含有效索引的列表。即本步骤中要考虑的样本。
        branch_name (string):   分支名称。['Root', 'Left', 'Right']
        max_depth (int):        生成树的最大深度。
        current_depth (int):    当前深度。递归调用期间使用的参数。""" 

    # 已达最大深度——停止划分
    if current_depth == max_depth:
        formatting = " "*current_depth + "-"*current_depth
        print(formatting, "%s leaf node with indices" % branch_name, node_indices)
        return
   
    # 否则，取最佳划分并划分数据
    # 获取该节点的最佳特征和阈值
    best_feature = get_best_split(X, y, node_indices) 
    tree.append((current_depth, branch_name, best_feature, node_indices))
    
    formatting = "-"*current_depth
    print("%s Depth %d, %s: Split on feature: %d" % (formatting, current_depth, branch_name, best_feature))
    
    # 按最佳特征划分数据集
    left_indices, right_indices = split_dataset(X, node_indices, best_feature)
    
    # 继续划分左子树和右子树。当前深度加一
    build_tree_recursive(X, y, left_indices, "Left", max_depth, current_depth+1)
    build_tree_recursive(X, y, right_indices, "Right", max_depth, current_depth+1)


In [ ]:
build_tree_recursive(X_train, y_train, root_indices, "Root", max_depth=2, current_depth=0)